# 06. 그룹별 요약 · `groupby` 정리 📦

지금까지 `.str` 로 **정제**하고 `.dt` 로 **파생컬럼**을 만들었습니다.
이제 마지막 단계 — 그 컬럼들을 기준으로 **묶어서 요약**할 차례입니다. 그게 `groupby` 예요.

`groupby` 는 딱 세 박자로 움직입니다 (**split → apply → combine**):

1. **split** : 기준 컬럼(예: 요일)으로 데이터를 그룹으로 쪼갠다
2. **apply** : 각 그룹에 계산(합계·평균·개수…)을 적용한다
3. **combine** : 결과를 하나의 표로 합친다

> 🔑 문법 : `df.groupby('기준컬럼')['계산할컬럼'].집계함수()`
> 예) `df.groupby('요일')['매출'].sum()` → **요일별 매출 합계**

`groupby` 는 08 연습문제에서 이미 만나봤죠. 오늘 개념을 확실히 매듭짓습니다.

## 0단계 · 어제까지의 정제를 한 번에 재현 (제공)
`.str`·`.dt` 로 만든 컬럼(`매출`, `요일`, `시간대`, `시도`, `도메인`, `아이스여부`)을 그대로 준비합니다. 실행만 하세요.

In [2]:
# [제공] 로드 + 정제/파생 (어제 내용 요약 재현)
import pandas as pd

df = pd.read_csv('data/주문로그.csv', encoding='utf-8-sig')

# --- 문자열 정제 (.str) ---
df['상품명']   = df['상품명'].str.strip().str.replace('ICE', '아이스').str.replace('ice', '아이스')
df['고객이메일'] = df['고객이메일'].str.strip().str.lower()
df['도메인']   = df['고객이메일'].str.split('@').str[1]
df['시도']     = df['주소'].str.split(' ').str[0]
df['가격_숫자'] = df['가격'].str.replace(',', '').str.replace('원', '').astype(int)
df['매출']     = df['가격_숫자'] * df['수량']
df['아이스여부'] = df['상품명'].str.startswith('아이스')

# --- 날짜 파생 (.dt) ---
df['주문일시'] = pd.to_datetime(df['주문일시'])
요일한글 = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
df['요일'] = df['주문일시'].dt.day_of_week.map(요일한글)
df['요일'] = pd.Categorical(df['요일'], categories=['월','화','수','목','금','토','일'], ordered=True)
df['시간대'] = pd.cut(df['주문일시'].dt.hour, bins=[0, 12, 17, 24],
                    labels=['오전', '오후', '저녁'], right=False)

print('준비 완료:', df.shape)
df.head()

준비 완료: (25, 14)


,주문번호,주문일시,고객이메일,상품명,가격,주소,수량,도메인,시도,가격_숫자,매출,아이스여부,요일,시간대
0,1,2025-03-13 08:15:00,kim.jy@gmail.com,아이스 아메리카노,"4,500원",서울특별시 강남구,2,gmail.com,서울특별시,4500,9000,True,목,오전
1,2,2025-03-13 09:40:00,lee2025@naver.com,카페라떼,"5,000원",서울특별시 마포구,1,naver.com,서울특별시,5000,5000,False,목,오전
2,3,2025-03-13 12:05:00,park_minsu@daum.net,아이스 카페라떼,"5,500원",경기도 성남시,1,daum.net,경기도,5500,5500,True,목,오후
3,4,2025-03-13 13:22:00,choi.aa@gmail.com,아메리카노,"4,500원",서울특별시 강남구,3,gmail.com,서울특별시,4500,13500,False,목,오후
4,5,2025-03-13 17:48:00,jung_h@kakao.com,아이스 바닐라라떼,"6,000원",서울특별시 송파구,1,kakao.com,서울특별시,6000,6000,True,목,저녁


## 🎯 미션 1 · 상품별 총매출 (단일 키 + 합계)

`상품명` 으로 묶어 `매출` 합계를 구하고, 많이 팔린 순으로 정렬하세요.
`df.groupby('상품명')['매출'].sum()` 뒤에 `.sort_values(ascending=False)` 를 붙입니다.

In [11]:
# [미션 1] 상품별 총매출 (내림차순)

# 👉 여기에 코드를 작성하세요
df_product_sales = df.groupby(by='상품명')['매출'].sum()\
                                .sort_values(ascending=False)\
                                .to_frame()

df_product_sales.to_excel('상품별 매출.xlsx')

## 🎯 미션 2 · 요일별 주문 건수 (그룹 크기 세기)

`요일` 로 묶어 **각 그룹에 몇 건이 있는지** 세보세요. 개수는 `.size()` 로 셉니다.
(요일은 월→일 순서로 정렬되도록 미리 지정해 뒀습니다.)

In [12]:
# [미션 2] 요일별 주문 건수
#  df.groupby('요일').size()

# 👉 여기에 코드를 작성하세요

df.groupby('요일').size()

요일
월    5
목    5
금    5
토    5
일    5
dtype: int64

## 🎯 미션 3 · 시간대별 매출 요약 (여러 통계 한 번에)

`시간대` 로 묶어 `매출` 의 **합계·평균·건수를 동시에** 구하세요.
`.agg(['sum', 'mean', 'count'])` 를 사용합니다.

In [18]:
# [미션 3] 시간대별 매출 합계·평균·건수

# 👉 여기에 코드를 작성하세요

result = (
    df.groupby('시간대')['매출']
      .agg(['sum', 'mean', 'count'])
      .rename(columns={
          'sum': '매출합계',
          'mean': '평균매출',
          'count': '총건수'
      })
      .round(2)
)
result


,매출합계,평균매출,총건수
시간대,,,
오전,90000,8181.82,11
오후,78500,8722.22,9
저녁,36000,7200.00,5


## 🎯 미션 4 · 지역 × 아이스여부 (다중 키)

기준을 **두 개** 줄 수도 있습니다. `시도` 와 `아이스여부` 로 묶어 `매출` 합계를 구하세요.
`df.groupby(['시도', '아이스여부'])['매출'].sum()`

In [14]:
# [미션 4] 시도 × 아이스여부 별 매출

# 👉 여기에 코드를 작성하세요

df.groupby(['시도', '아이스여부'])['매출'].sum().to_frame()

매출
시도    아이스여부       
경기도   False  18500
      True   31000
서울특별시 False  34500
      True   99500
인천광역시 False  21000

## 🎯 미션 5 · 도메인별 평균 결제금액 Top

고객이 쓰는 이메일 `도메인` 별로 `매출` 평균을 구해 높은 순으로 정렬하세요.
어떤 도메인 고객의 객단가가 높은지 봅니다.

In [19]:
# [미션 5] 도메인별 평균 매출 (내림차순)

# 👉 여기에 코드를 작성하세요

df.groupby('도메인')['매출'].mean().sort_values(ascending=False)

도메인
gmail.com    9000.000000
daum.net     8600.000000
naver.com    8428.571429
kakao.com    6100.000000
Name: 매출, dtype: float64

## ✅ 정리 · "가공 → 요약" 한 흐름을 완성했습니다

오늘 3개 노트북을 하나로 이으면 이렇게 됩니다.

```
지저분한 원본  →  .str 정제  →  .dt 날짜 파생  →  groupby 요약  →  인사이트
```

방금 만든 표들로 이런 질문에 답할 수 있어요:
- 어떤 메뉴가 매출 1위인가? (미션 1)
- 어느 요일·시간대에 주문이 몰리나? (미션 2·3)
- 지역·고객군별로 소비 패턴이 다른가? (미션 4·5)

> 💡 핵심 : **분류할 컬럼을 잘 만들어 두면(.str/.dt), 요약은 `groupby` 한 줄이면 끝**입니다.
> 그래서 앞 단계(정제·파생)가 중요했던 거예요.

---
### 💡 `groupby` 한눈에

| 코드 | 하는 일 |
|------|---------|
| `df.groupby('A')['B'].sum()` | A별 B 합계 |
| `df.groupby('A')['B'].mean()` | A별 B 평균 |
| `df.groupby('A').size()` | A별 행 개수 |
| `df.groupby('A')['B'].agg(['sum','mean','count'])` | 여러 통계 동시 |
| `df.groupby(['A','C'])['B'].sum()` | 두 기준으로 묶기 |
| `... .sort_values(ascending=False)` | 큰 값 순 정렬 |
